# 01 · Clean TSAC
**Input:** `data/raw/tsac/{train,test}_{pos,neg}.txt` · **Output:** `data/interim/tsac_clean.csv`, `logs/cleaning_log_tsac.csv`

TSAC holds our only Arabic-script comments **and** our test set, so every step here protects the test set.
Cleaning removes rows and fixes encoding only: no lowercasing, no rewriting words.

In [1]:
# Works both locally (repo root or notebooks/) and in Colab after cloning the repo.
import os, sys
from pathlib import Path
if Path("notebooks").is_dir():
    ROOT = Path(".").resolve()
elif Path("../notebooks").is_dir():
    ROOT = Path("..").resolve()
else:  # Colab: the repository is cloned into /content first
    ROOT = Path("/content/tunisian-sentiment")
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
print("Repo root found:", ROOT.name)

Repo root found: tunisian-sentiment


In [2]:
import pandas as pd
from src.load_raw import load_tsac
from src.cleaning_utils import fix_text, has_letter, CleaningLog

OUT = ROOT / "data/interim/tsac_clean.csv"
LOG = ROOT / "logs/cleaning_log_tsac.csv"
log = CleaningLog()

def by_split(d):
    """'train N / test M' for the log notes."""
    c = d.split.value_counts()
    return f"train {c.get('train', 0)} / test {c.get('test', 0)}"

## 1. Load train and test, assign ids
Each split comes as two files, one per label, so the label comes from the file name (`pos` / `neg`) and needs no mapping.
Ids come from the shared loader (`src/load_raw.py`): per split, positives first, then negatives.

In [3]:
df = pd.concat([load_tsac("train", ROOT), load_tsac("test", ROOT)], ignore_index=True)
log.add("load", len(df), len(df), by_split(df))
print(pd.crosstab(df.split, df.label, margins=True))
df.sample(5, random_state=42)

load                                            17069 ->   17069  (removed 0)


label   neg   pos    All
split                   
test   1700  1700   3400
train  6515  7154  13669
All    8215  8854  17069


,id,text,label,split
3069,tsac_train_03070,"Mash'Allah 3lliha , rabbi ysoun w yberik :*",pos,train
10508,tsac_train_10509,راك ماسط براشوف ما يصلح بيك,neg,train
13030,tsac_train_13031,لعنة الله عليكم .. يا مفسدين يا كلاب .. تخيبوا...,neg,train
16666,tsac_test_02998,لايليق بهذا البرنامج,neg,test
13976,tsac_test_00308,Mahlekoum rab ya5lkoum le ba3thkoum,pos,test


## 2. Missing values and encoding artifacts
A missing comment shows up as an empty line. `fix_text()` replaces the `_x000D_` artifacts (a Windows line-break code left over from an Excel export) with a space, then strips and collapses spaces.

In [4]:
print("NaN per column:\n", df.isna().sum())
n_x = df.text.str.contains("_x000D_", regex=False).sum()
print(df[df.text.str.contains("_x000D_", regex=False)].text.head(3).tolist())
df["text"] = df.text.map(fix_text)
log.add("fix_text (_x000D_, spaces)", len(df), len(df), f"{n_x} rows had _x000D_")

NaN per column:
 id       0
text     0
label    0
split    0
dtype: int64
['هشام الخلصي نحبوا من قبل و زيد ظهر إحب الترجي حبيتوا أكثر #bravo_hichem :-*  _x000D_', '"ههههههههههه والله حلقة عالمية _x000D_', '"Ya w7ayed 5aterha bentek_x000D_']
fix_text (_x000D_, spaces)                      17069 ->   17069  (removed 0)


## 3. Empty rows and rows without any letter
Rows of only emoji, digits or punctuation carry no text the conversion can work on.

In [5]:
before = len(df); df = df[df.text != ""]
log.add("remove empty rows", before, len(df), by_split(df))
nolet = ~df.text.map(has_letter)
print("Examples without letters:", df[nolet].text.head(8).tolist())
before = len(df); df = df[~nolet]
log.add("remove rows without any letter", before, len(df), by_split(df))

remove empty rows                               17069 ->   17061  (removed 8)
Examples without letters: ['♥♥♥', '.😍😍😍😍😘😘😘😘😘😳😳', '😉😘😘😘', '😂😂😂😉😉😉😘😘😘', '"', '<3', '😍😘💙❤💖💕💓💜💛💚💟💞💝💗💕', '!!!"']
remove rows without any letter                  17061 ->   16994  (removed 67)


## 4. Label conflicts
The same text with **both** labels anywhere in TSAC: we cannot know the true label, so all copies are removed.

In [6]:
n_labels = df.groupby("text").label.transform("nunique")
conflicts = df[n_labels > 1]
print(conflicts.sort_values("text")[["id", "split", "label", "text"]].head(10).to_string())
before = len(df); df = df[n_labels == 1]
log.add("remove label conflicts (all copies)", before, len(df), f"{conflicts.text.nunique()} distinct texts")

                     id  split label   text
503    tsac_train_00504  train   pos  3asla
14148   tsac_test_00480   test   pos  3asla
14009   tsac_test_00341   test   pos  3asla
4086   tsac_train_04087  train   pos  3asla
6231   tsac_train_06232  train   pos  3asla
4018   tsac_train_04019  train   pos  3asla
5969   tsac_train_05970  train   pos  3asla
5365   tsac_train_05366  train   pos  3asla
12423  tsac_train_12424  train   neg  3asla
14627   tsac_test_00959   test   pos  3asla
remove label conflicts (all copies)             16994 ->   16887  (removed 107)


## 5. Duplicates inside one split
Same text twice in train (or twice in test): keep the first copy.

In [7]:
dup_texts_total = df.text.duplicated().sum()
print("Duplicate rows across the whole corpus (before steps 5-6):", dup_texts_total)
before = len(df); df = df.drop_duplicates(subset=["split", "text"], keep="first")
log.add("remove duplicates inside a split (keep first)", before, len(df), by_split(df))

Duplicate rows across the whole corpus (before steps 5-6): 1686
remove duplicates inside a split (keep first)   16887 ->   15513  (removed 1374)


## 6. Leakage between train and test
A text in both train and test would let a model "see" a test comment during training. We **remove it from train and keep it in test**: the official test set stays intact, so our scores stay comparable with the TSAC paper; train loses about 2%.

In [8]:
test_texts = set(df.loc[df.split == "test", "text"])
leak = (df.split == "train") & df.text.isin(test_texts)
before = len(df); df = df[~leak]
log.add("remove train copies of test texts (leakage)", before, len(df), by_split(df))
assert not set(df.loc[df.split == "train", "text"]) & set(df.loc[df.split == "test", "text"])

remove train copies of test texts (leakage)     15513 ->   15201  (removed 312)


## 7. Length: no cutoff
Unlike TUNIZI, TSAC is not cut by length: the test set must stay as published apart from duplicates.

In [9]:
print(df.text.str.split().str.len().describe(percentiles=[.5, .95, .99]).round(1))

count    15201.0
mean         7.0
std          8.8
min          1.0
50%          4.0
95%         21.0
99%         41.0
max        208.0
Name: text, dtype: float64


## 8. Add `source`, check, save

In [10]:
df["source"] = "tsac"
df = df[["id", "text", "label", "source", "split"]]
assert set(df.label) <= {"pos", "neg"} and df.id.is_unique and (df.text != "").all()
OUT.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(OUT, index=False, encoding="utf-8")
log.add("final tsac_clean.csv", len(df), len(df), by_split(df))
log.save(LOG)
print(pd.crosstab(df.split, df.label, margins=True))
log.to_frame()

final tsac_clean.csv                            15201 ->   15201  (removed 0)
label   neg   pos    All
split                   
test   1626  1582   3208
train  5712  6281  11993
All    7338  7863  15201


,step,rows_before,rows_after,removed,note
0,load,17069,17069,0,train 13669 / test 3400
1,"fix_text (_x000D_, spaces)",17069,17069,0,260 rows had _x000D_
2,remove empty rows,17069,17061,8,train 13662 / test 3399
3,remove rows without any letter,17061,16994,67,train 13612 / test 3382
4,remove label conflicts (all copies),16994,16887,107,12 distinct texts
5,remove duplicates inside a split (keep first),16887,15513,1374,train 12305 / test 3208
6,remove train copies of test texts (leakage),15513,15201,312,train 11993 / test 3208
7,final tsac_clean.csv,15201,15201,0,train 11993 / test 3208
